In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
df = pd.read_csv("fake_and_real_news.csv")
df.head()

,Text,label
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake
1,U.S. conservative leader optimistic of common ...,Real
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real
3,Court Forces Ohio To Allow Millions Of Illega...,Fake
4,Democrats say Trump agrees to work on immigrat...,Real


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9900 entries, 0 to 9899
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Text    9900 non-null   object
 1   label   9900 non-null   object
dtypes: object(2)
memory usage: 154.8+ KB


In [4]:
#Encoding labels
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df["label"] = encoder.fit_transform(df["label"])
df.head()

,Text,label
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,0
1,U.S. conservative leader optimistic of common ...,1
2,"Trump proposes U.S. tax overhaul, stirs concer...",1
3,Court Forces Ohio To Allow Millions Of Illega...,0
4,Democrats say Trump agrees to work on immigrat...,1


In [5]:
df["Text"] = df["Text"].fillna("").astype(str)
X=df["Text"]

news = X.tolist()
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(news)

word_index = tokenizer.word_index
vocab_size = len(word_index)
sequences = tokenizer.texts_to_sequences(news)
X = pad_sequences(sequences,maxlen=500,padding='post',truncating='post')

y=df["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # Proportion of the dataset to use for testing (e.g., 20%)
    random_state=42,    # Ensures reproducibility
    stratify=y          # Ensures equal/proportional split based on labels
)

In [6]:
#Embedding
embedding_index={}
embedding_dim = 50
with open ("glove.6B.50d.txt",'r',encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:],dtype='float32')
        embedding_index[word] = coefs
    embedding_matrix = np.zeros((vocab_size, embedding_dim))
    for word,i in word_index.items():
        if i<vocab_size:
            embedding_vector=embedding_index.get(word)
            if embedding_vector is not None:
                embedding_matrix[i]=embedding_vector

In [7]:
num_classes = len(np.unique(y_train))
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=False
    ),
    tf.keras.layers.SpatialDropout1D(0.2),
    tf.keras.layers.Conv1D(
        filters=128,
        kernel_size=5,
        activation="relu",
        padding="same"
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling1D(pool_size=2),
    tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(128, return_sequences=True)
    ),
    tf.keras.layers.GlobalMaxPooling1D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(
        num_classes,
        activation="softmax"
    )
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │       3,443,300 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ spatial_dropout1d (SpatialDropout1D) │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d (Conv1D)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ ?                           │     0 (unbuilt) │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d (MaxPooling1D)         │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional (Bidirectional)        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_max_pooling1d                 │ ?                           │               0 │
│ (GlobalMaxPooling1D)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,443,300 (13.14 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 3,443,300 (13.14 MB)

In [8]:
history = model.fit(
    np.array(X_train), 
    np.array(y_train), 
    epochs=5, 
    validation_data=(np.array(X_test), np.array(y_test)), 
    verbose=2
)

Epoch 1/5
248/248 - 63s - 253ms/step - accuracy: 0.9324 - loss: 0.1656 - val_accuracy: 0.9879 - val_loss: 0.0370
Epoch 2/5
248/248 - 59s - 236ms/step - accuracy: 0.9878 - loss: 0.0336 - val_accuracy: 0.9985 - val_loss: 0.0055
Epoch 3/5
248/248 - 59s - 240ms/step - accuracy: 0.9951 - loss: 0.0157 - val_accuracy: 0.9990 - val_loss: 0.0037
Epoch 4/5
248/248 - 60s - 241ms/step - accuracy: 0.9980 - loss: 0.0062 - val_accuracy: 1.0000 - val_loss: 9.2784e-04
Epoch 5/5
248/248 - 60s - 241ms/step - accuracy: 0.9985 - loss: 0.0050 - val_accuracy: 1.0000 - val_loss: 7.6455e-04


In [9]:
from sklearn.metrics import classification_report
import numpy as np

predictions = model.predict(X_test)
predicted_classes = np.argmax(predictions, axis=1)

print(classification_report(
    np.array(y_test),
    predicted_classes,
    target_names=encoder.classes_
))

62/62 ━━━━━━━━━━━━━━━━━━━━ 5s 74ms/step
              precision    recall  f1-score   support

        Fake       1.00      1.00      1.00      1000
        Real       1.00      1.00      1.00       980

    accuracy                           1.00      1980
   macro avg       1.00      1.00      1.00      1980
weighted avg       1.00      1.00      1.00      1980



In [10]:
loss, accuracy = model.evaluate(
    X_test,
    np.array(y_test),
    verbose=0
)

print("Test Accuracy:", accuracy)

Test Accuracy: 1.0


In [11]:
model.save("news_verify_model.keras")

In [12]:
import pickle
with open("tokenizer_v.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [13]:
with open("label_encoder_v.pkl", "wb") as f:
    pickle.dump(encoder, f)